# FreshMart Day 2 — Practice: clean data → live dashboards

**The story:** FreshMart has 3 branches — Ikeja, Wuse and Garki. Every week each branch sends a messy sales file.
Our script `clean_sales.py` already cleans them. In this practice you make the script **update dashboards by itself**
and **email you** every time something changes.

**Before you start this notebook:** finish Parts 1, 2 and 3 of `STUDENT_GUIDE.pdf`.
You need `credentials.json` in this folder and a Google Sheet called **FreshMart Dashboard**.

**How to work:** run each cell with **Shift + Enter**. Where you see `___`, replace it with the right code.
Stuck? The answers are at the end of the guide. Try first!

## Step 0 — Get ready

In [ ]:
import gspread
import pandas as pd
print("Ready!")

## Step 1 — Re-use the cleaning
You don't copy the cleaning code. You **import** it from `clean_sales.py`, the same way you import pandas.
`load_all_files()` reads every file in `data/raw`, `clean()` fixes the 6 problems, and `summarise()` builds the summary tables.

In [ ]:
from clean_sales import load_all_files, clean, summarise

sales = clean(load_all_files())
summaries = summarise(___)      # which table do we summarise?
summaries["by_branch"]

✅ You should see 3 branches, with **Ikeja on top (757,400)**.

## Step 2 — Connect to Google Sheets
Line 1 logs in as your robot using the key file. Line 2 opens your Sheet by its name.

In [ ]:
gc = gspread.service_account(filename=___)      # the name of your key file, in quotes
sheet = gc.open(___)                            # the name of your Google Sheet, in quotes
print("Connected to:", sheet.title)
print(sheet.url)

✅ You should see `Connected to: FreshMart Dashboard` and a link. Click the link and keep the Sheet open.

## Step 3 — Send the clean data
Google Sheets doesn't understand pandas tables. It wants a **list of rows**: the column names first, then every row as text.

In [ ]:
rows = [sales.columns.___()] + sales.astype(str).values.tolist()    # turn the column names into a list
rows[:3]

✅ You should see 3 small lists: the column names, then two sales. Now run the next cell as it is.

In [ ]:
def get_tab(sheet, name):
    """Open a tab by name, or create it if it doesn't exist yet."""
    try:
        return sheet.worksheet(name)
    except gspread.WorksheetNotFound:
        return sheet.add_worksheet(title=name, rows=1000, cols=20)

tab = get_tab(sheet, "Clean Data")
tab.clear()
tab.update(values=rows, range_name="A1", value_input_option="USER_ENTERED")
print("Sent", len(rows) - 1, "rows")

👀 **Look at your Google Sheet now.** A new tab called *Clean Data* has 66 rows. Python wrote them.

## Step 4 — Send the summaries (one tab each)

In [ ]:
def send_table(sheet, tab_name, table):
    rows = [table.columns.tolist()] + table.astype(str).values.tolist()
    tab = get_tab(sheet, tab_name)
    tab.clear()
    tab.update(values=rows, range_name="A1", value_input_option="USER_ENTERED")
    print("Sent", tab_name)

send_table(sheet, "By Branch", summaries["by_branch"])
# Send by_product to a tab called "By Product", by_day to "By Day", and by_payment to "By Payment" - the same way
___
___
___

✅ Your Sheet now has the tabs **By Branch, By Product, By Day, By Payment**.

**Now go to the guide, Part 5,** and build 3 charts in Google Sheets. Come back here when you're done.

---
## Step 5 — Power BI
Run the cell below first. It makes sure `output/freshmart_clean_latest.xlsx` (the file Power BI reads) is fresh.
Then follow **Part 6** of the guide.

In [ ]:
!python clean_sales.py

✅ The last line says **Done: 66 clean rows, total revenue ₦1,980,250**.

---
## Step 6 — n8n: get told when it runs
Python does the work. **n8n Cloud** sends you the news. After every run, the script sends a short message to an
n8n **webhook** (a web address that receives data). n8n then emails you.

Follow **Part 7** of the guide. When it tells you to, run the cell below.

In [ ]:
!python update_dashboard.py

✅ You should see `Told n8n: success (n8n replied 200)` near the end.

---
## Step 7 — The magic moment ✨
Follow **Part 8** of the guide to start the scheduler. When the scheduler is running, run the cell below:
a **new branch, Lekki**, sends its first file.

In [ ]:
import shutil
shutil.copy("data/new_week/lekki_week41.xlsx", "data/raw/")
print("Lekki file added to data/raw - now WAIT 1 minute, don't touch anything")

Within 1 minute, without you pressing anything:
- 👀 **Google Sheets:** Lekki appears in the branch chart (4 bars), and "Last updated" in cell D1 changes
- 📧 **Your email:** *FreshMart dashboard updated - ₦2,674,950*
- 👀 **Power BI:** click **Home → Refresh**, and Lekki appears there too

You changed **zero** lines of code.

---
## Step 8 — When things break, you get told
A manager sent a damaged file. Keep the scheduler running and run this cell:

In [ ]:
shutil.copy("data/new_week/broken_file_week41.xlsx", "data/raw/")
print("Broken file added - wait 1 minute for the alert")

📧 Within 1 minute: an email **FreshMart script FAILED** with the error. Now remove the bad file:

In [ ]:
import os
os.remove("data/raw/broken_file_week41.xlsx")
print("Broken file removed")

The next run works again. No new email comes, because the numbers didn't change.
Go to the scheduler window and press **Ctrl + C** to stop it. **Well done — you built the whole pipeline!**

---
## Practise again from the start
Run this cell to put the folder back the way it was (it removes Lekki and the broken file).
Then put `scheduler.py` back to the cleaning line (guide, *Practise again*).

In [ ]:
import os
for f in ["data/raw/lekki_week41.xlsx", "data/raw/broken_file_week41.xlsx", "output/last_run.json"]:
    if os.path.exists(f):
        os.remove(f)
print("Reset done")